# Exploratory Data Analysis: Financial News Dataset
## Understanding our data before sentiment analysis

**Notebook Objective:** Explore the collected financial news data, understand its structure, identify patterns, and prepare insights for sentiment analysis.

**Data Source:** Raw news collected from NewsAPI and RSS feeds
**Date:** March 2026

## 1. Setup and Imports

In [12]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)



## 2. Load the Processed Data

In [17]:
# Define data path
PROCESSED_DATA_PATH = "notebooks/processed/processed_financial_news.csv"
RAW_DATA_PATH = "notebooks/raw_news_data.csv"

# Check if processed data exists, otherwise load raw
if os.path.exists(PROCESSED_DATA_PATH):
    print(f"Loading processed data from: {PROCESSED_DATA_PATH}")
    df = pd.read_csv(PROCESSED_DATA_PATH)
    data_source = "processed"
elif os.path.exists(RAW_DATA_PATH):
    print(f"Processed data not found. Loading raw data from: {RAW_DATA_PATH}")
    df = pd.read_csv(RAW_DATA_PATH)
    data_source = "raw"
    print("\nNote: For full EDA features, please run preprocessing first.")
else:
    print("No data files found. Please run the collector first.")
    df = pd.DataFrame()

if not df.empty:
    print(f"\nDataset shape: {df.shape}")
    print(f"Number of articles: {len(df)}")
    print(f"Number of features: {len(df.columns)}")

No data files found. Please run the collector first.


## 3. Initial Data Overview

In [ ]:
# Display first few rows
print("First 5 rows of the dataset:")
df.head()

In [ ]:
# Data types and info
print("Dataset Information:")
print("-" * 50)
df.info()

In [ ]:
# Basic statistics for numerical columns
print("Statistical Summary for Numerical Columns:")
print("-" * 50)
df.describe()

## 4. Missing Data Analysis

In [ ]:
# Calculate missing values
missing_data = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isnull().sum().values,
    'missing_percentage': (df.isnull().sum().values / len(df)) * 100
})
missing_data = missing_data[missing_data['missing_count'] > 0].sort_values('missing_count', ascending=False)

print("Missing Data Summary:")
print("-" * 50)
if len(missing_data) > 0:
    print(missing_data.to_string(index=False))
else:
    print("No missing values found!")

# Visualize missing data
if len(missing_data) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(missing_data['column'], missing_data['missing_percentage'])
    plt.xlabel('Missing Percentage (%)')
    plt.title('Missing Data by Column')
    plt.tight_layout()
    plt.show()

## 5. Source Distribution Analysis

In [ ]:
# Distribution by collection method
if 'collection_method' in df.columns:
    method_counts = df['collection_method'].value_counts()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    axes[0].bar(method_counts.index, method_counts.values, color=['blue', 'orange'])
    axes[0].set_xlabel('Collection Method')
    axes[0].set_ylabel('Number of Articles')
    axes[0].set_title('Articles by Collection Method')
    
    # Add value labels
    for i, v in enumerate(method_counts.values):
        axes[0].text(i, v + 5, str(v), ha='center')
    
    # Pie chart
    axes[1].pie(method_counts.values, labels=method_counts.index, autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Collection Method Distribution')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Top news sources
if 'source' in df.columns:
    source_counts = df['source'].value_counts().head(15)
    
    plt.figure(figsize=(12, 6))
    plt.barh(range(len(source_counts)), source_counts.values)
    plt.yticks(range(len(source_counts)), source_counts.index)
    plt.xlabel('Number of Articles')
    plt.title('Top 15 News Sources')
    
    # Add value labels
    for i, v in enumerate(source_counts.values):
        plt.text(v + 1, i, str(v), va='center')
    
    plt.tight_layout()
    plt.show()
    
    print("\nTop 10 Sources by Article Count:")
    print(source_counts.head(10).to_string())

## 6. Temporal Analysis

In [ ]:
# Convert published date to datetime
if 'published' in df.columns:
    df['published_dt'] = pd.to_datetime(df['published'], errors='coerce')
    df['date'] = df['published_dt'].dt.date
    df['hour'] = df['published_dt'].dt.hour
    df['day_of_week'] = df['published_dt'].dt.day_name()
    df['month'] = df['published_dt'].dt.month_name()
    
    # Remove rows with invalid dates
    valid_dates = df['published_dt'].notna()
    print(f"Articles with valid dates: {valid_dates.sum()} out of {len(df)}")
    
    if valid_dates.sum() > 0:
        df_temp = df[valid_dates].copy()
        
        # Time series plot
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Articles over time
        daily_counts = df_temp.groupby('date').size()
        axes[0, 0].plot(range(len(daily_counts)), daily_counts.values, marker='o', linestyle='-', markersize=3)
        axes[0, 0].set_xlabel('Day')
        axes[0, 0].set_ylabel('Number of Articles')
        axes[0, 0].set_title('Articles Collected Over Time')
        axes[0, 0].tick_params(axis='x', rotation=45)
        
        # Hour of day distribution
        hour_counts = df_temp['hour'].value_counts().sort_index()
        axes[0, 1].bar(hour_counts.index, hour_counts.values)
        axes[0, 1].set_xlabel('Hour of Day (24h)')
        axes[0, 1].set_ylabel('Number of Articles')
        axes[0, 1].set_title('Articles by Hour of Day')
        
        # Day of week distribution
        day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
        day_counts = df_temp['day_of_week'].value_counts().reindex(day_order)
        axes[1, 0].bar(day_counts.index, day_counts.values)
        axes[1, 0].set_xlabel('Day of Week')
        axes[1, 0].set_ylabel('Number of Articles')
        axes[1, 0].set_title('Articles by Day of Week')
        axes[1, 0].tick_params(axis='x', rotation=45)
        
        # Month distribution
        month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                       'July', 'August', 'September', 'October', 'November', 'December']
        month_counts = df_temp['month'].value_counts().reindex(month_order)
        axes[1, 1].bar(month_counts.index, month_counts.values)
        axes[1, 1].set_xlabel('Month')
        axes[1, 1].set_ylabel('Number of Articles')
        axes[1, 1].set_title('Articles by Month')
        axes[1, 1].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

## 7. Text Length Analysis

In [ ]:
# Check for text columns
text_columns = ['cleaned_text', 'content', 'description', 'text']
available_text_col = None

for col in text_columns:
    if col in df.columns:
        available_text_col = col
        print(f"Using '{col}' for text analysis")
        break

if available_text_col:
    # Calculate text lengths if not already present
    if 'word_count' not in df.columns:
        df['word_count'] = df[available_text_col].astype(str).apply(lambda x: len(x.split()))
    if 'char_count' not in df.columns:
        df['char_count'] = df[available_text_col].astype(str).apply(len)
    
    # Text length statistics
    print("\nText Length Statistics:")
    print("-" * 50)
    print(f"Average word count: {df['word_count'].mean():.1f}")
    print(f"Median word count: {df['word_count'].median():.1f}")
    print(f"Minimum word count: {df['word_count'].min()}")
    print(f"Maximum word count: {df['word_count'].max()}")
    print(f"Standard deviation: {df['word_count'].std():.1f}")
    
    # Visualize word count distribution
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Histogram
    axes[0].hist(df['word_count'], bins=30, edgecolor='black', alpha=0.7)
    axes[0].axvline(df['word_count'].mean(), color='red', linestyle='--', label=f"Mean: {df['word_count'].mean():.1f}")
    axes[0].axvline(df['word_count'].median(), color='green', linestyle='--', label=f"Median: {df['word_count'].median():.1f}")
    axes[0].set_xlabel('Word Count')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Article Length')
    axes[0].legend()
    
    # Box plot
    axes[1].boxplot(df['word_count'])
    axes[1].set_ylabel('Word Count')
    axes[1].set_title('Box Plot of Article Length')
    
    # Cumulative distribution
    sorted_lengths = np.sort(df['word_count'])
    yvals = np.arange(len(sorted_lengths)) / float(len(sorted_lengths))
    axes[2].plot(sorted_lengths, yvals)
    axes[2].axhline(y=0.5, color='red', linestyle='--', label='Median')
    axes[2].axhline(y=0.8, color='orange', linestyle='--', label='80th percentile')
    axes[2].set_xlabel('Word Count')
    axes[2].set_ylabel('Cumulative Probability')
    axes[2].set_title('Cumulative Distribution')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
    
    # Articles by length category
    df['length_category'] = pd.cut(df['word_count'], 
                                   bins=[0, 50, 100, 200, 500, 1000, float('inf')],
                                   labels=['Very Short (<50)', 'Short (50-100)', 
                                           'Medium (100-200)', 'Long (200-500)', 
                                           'Very Long (500-1000)', 'Extremely Long (>1000)'])
    
    length_dist = df['length_category'].value_counts().sort_index()
    
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(length_dist)), length_dist.values)
    plt.xticks(range(len(length_dist)), length_dist.index, rotation=45, ha='right')
    plt.xlabel('Length Category')
    plt.ylabel('Number of Articles')
    plt.title('Articles by Length Category')
    
    for i, v in enumerate(length_dist.values):
        plt.text(i, v + 5, str(v), ha='center')
    
    plt.tight_layout()
    plt.show()

## 8. Financial Content Analysis

In [ ]:
# Check for financial content indicators
financial_columns = ['financial_term_count', 'money_mentions', 'percentage_mentions', 'has_financial_content']
available_fin_cols = [col for col in financial_columns if col in df.columns]

if available_fin_cols:
    print("Financial Content Analysis:")
    print("-" * 50)
    
    if 'has_financial_content' in df.columns:
        fin_count = df['has_financial_content'].sum()
        fin_pct = (fin_count / len(df)) * 100
        print(f"Articles with financial terms: {fin_count} ({fin_pct:.1f}%)")
    
    if 'financial_term_count' in df.columns:
        print(f"\nFinancial Terms Statistics:")
        print(f"  Mean: {df['financial_term_count'].mean():.2f}")
        print(f"  Median: {df['financial_term_count'].median():.2f}")
        print(f"  Max: {df['financial_term_count'].max()}")
    
    if 'money_mentions' in df.columns:
        money_count = (df['money_mentions'] > 0).sum()
        money_pct = (money_count / len(df)) * 100
        print(f"\nArticles with money mentions: {money_count} ({money_pct:.1f}%)")
    
    if 'percentage_mentions' in df.columns:
        pct_count = (df['percentage_mentions'] > 0).sum()
        pct_pct = (pct_count / len(df)) * 100
        print(f"Articles with percentage mentions: {pct_count} ({pct_pct:.1f}%)")
    
    # Visualize financial content
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    if 'has_financial_content' in df.columns:
        # Pie chart for financial content
        fin_pie = df['has_financial_content'].value_counts()
        axes[0, 0].pie(fin_pie.values, labels=['Has Financial Terms', 'No Financial Terms'], 
                       autopct='%1.1f%%', colors=['green', 'gray'])
        axes[0, 0].set_title('Financial Content Distribution')
    
    if 'financial_term_count' in df.columns:
        # Histogram of financial terms
        axes[0, 1].hist(df[df['financial_term_count'] > 0]['financial_term_count'], bins=20, edgecolor='black', alpha=0.7)
        axes[0, 1].set_xlabel('Number of Financial Terms')
        axes[0, 1].set_ylabel('Frequency')
        axes[0, 1].set_title('Distribution of Financial Terms')
    
    if 'money_mentions' in df.columns:
        # Money mentions
        axes[1, 0].hist(df[df['money_mentions'] > 0]['money_mentions'], bins=10, edgecolor='black', alpha=0.7, color='green')
        axes[1, 0].set_xlabel('Number of Money Mentions')
        axes[1, 0].set_ylabel('Frequency')
        axes[1, 0].set_title('Articles with Money Mentions')
    
    if 'percentage_mentions' in df.columns:
        # Percentage mentions
        axes[1, 1].hist(df[df['percentage_mentions'] > 0]['percentage_mentions'], bins=10, edgecolor='black', alpha=0.7, color='orange')
        axes[1, 1].set_xlabel('Number of Percentage Mentions')
        axes[1, 1].set_ylabel('Frequency')
        axes[1, 1].set_title('Articles with Percentage Mentions')
    
    plt.tight_layout()
    plt.show()

## 9. Data Quality Assessment

In [ ]:
# Check for quality score
if 'data_quality_score' in df.columns:
    print("Data Quality Score Analysis:")
    print("-" * 50)
    print(f"Mean quality score: {df['data_quality_score'].mean():.3f}")
    print(f"Median quality score: {df['data_quality_score'].median():.3f}")
    print(f"Min quality score: {df['data_quality_score'].min():.3f}")
    print(f"Max quality score: {df['data_quality_score'].max():.3f}")
    
    # Quality score distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(df['data_quality_score'], bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(df['data_quality_score'].mean(), color='red', linestyle='--', 
                    label=f"Mean: {df['data_quality_score'].mean():.3f}")
    axes[0].set_xlabel('Quality Score')
    axes[0].set_ylabel('Frequency')
    axes[0].set_title('Distribution of Data Quality Scores')
    axes[0].legend()
    
    # Quality categories
    df['quality_category'] = pd.cut(df['data_quality_score'], 
                                    bins=[0, 0.3, 0.5, 0.7, 1.0],
                                    labels=['Low (<0.3)', 'Medium (0.3-0.5)', 
                                            'Good (0.5-0.7)', 'Excellent (>0.7)'])
    
    quality_dist = df['quality_category'].value_counts().sort_index()
    axes[1].bar(quality_dist.index, quality_dist.values)
    axes[1].set_xlabel('Quality Category')
    axes[1].set_ylabel('Number of Articles')
    axes[1].set_title('Articles by Quality Category')
    axes[1].tick_params(axis='x', rotation=45)
    
    for i, v in enumerate(quality_dist.values):
        axes[1].text(i, v + 5, str(v), ha='center')
    
    plt.tight_layout()
    plt.show()
    
    # High quality articles
    if 'ready_for_analysis' in df.columns:
        ready_count = df['ready_for_analysis'].sum()
        ready_pct = (ready_count / len(df)) * 100
        print(f"\nArticles ready for sentiment analysis: {ready_count} ({ready_pct:.1f}%)")
    else:
        # Define readiness based on quality score and word count
        ready = (df['data_quality_score'] > 0.5) & (df['word_count'] >= 20)
        ready_count = ready.sum()
        ready_pct = (ready_count / len(df)) * 100
        print(f"\nArticles suitable for analysis (quality>0.5, words>=20): {ready_count} ({ready_pct:.1f}%)")

## 10. Correlation Analysis

In [ ]:
# Select numerical columns for correlation
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numerical_cols) > 1:
    # Calculate correlation matrix
    corr_matrix = df[numerical_cols].corr()
    
    # Plot heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
    plt.title('Correlation Matrix of Numerical Features')
    plt.tight_layout()
    plt.show()
    
    # Find strong correlations
    print("Strong Correlations (|r| > 0.5):")
    print("-" * 50)
    
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.5:
                print(f"  {corr_matrix.columns[i]} vs {corr_matrix.columns[j]}: {corr_matrix.iloc[i, j]:.3f}")

## 11. Word Cloud Visualization

In [ ]:
# Try to import wordcloud (optional)
try:
    from wordcloud import WordCloud, STOPWORDS
    wordcloud_available = True
except ImportError:
    wordcloud_available = False
    print("WordCloud library not installed. Install with: pip install wordcloud")

if wordcloud_available and available_text_col:
    # Combine text from high quality articles
    if 'data_quality_score' in df.columns:
        high_quality = df[df['data_quality_score'] > 0.5]
        text_data = high_quality[available_text_col].dropna().astype(str)
    else:
        text_data = df[available_text_col].dropna().astype(str)
    
    if len(text_data) > 0:
        all_text = ' '.join(text_data)
        
        # Create word cloud
        wordcloud = WordCloud(
            width=1200,
            height=600,
            background_color='white',
            max_words=150,
            colormap='viridis',
            stopwords=set(STOPWORDS)
        ).generate(all_text)
        
        plt.figure(figsize=(15, 8))
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title('Most Common Words in Financial News', fontsize=16, pad=20)
        plt.show()

## 12. Summary Statistics by Source

In [ ]:
if 'source' in df.columns and 'word_count' in df.columns:
    # Group by source
    source_stats = df.groupby('source').agg({
        'title': 'count',
        'word_count': ['mean', 'median', 'max'],
        'financial_term_count': ['mean', 'sum'] if 'financial_term_count' in df.columns else None
    }).round(1)
    
    source_stats.columns = ['_'.join(col).strip() for col in source_stats.columns.values]
    source_stats = source_stats.rename(columns={'title_count': 'article_count'})
    source_stats = source_stats.sort_values('article_count', ascending=False).head(10)
    
    print("Top 10 Sources - Statistics:")
    print("-" * 70)
    print(source_stats.to_string())

## 13. Dataset Summary Report

In [ ]:
print("=" * 70)
print("DATASET SUMMARY REPORT")
print("=" * 70)

print(f"\n📊 BASIC INFORMATION:")
print(f"   Total articles: {len(df)}")
print(f"   Total features: {len(df.columns)}")
print(f"   Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

if 'collection_method' in df.columns:
    print(f"\n📡 DATA SOURCES:")
    for method, count in df['collection_method'].value_counts().items():
        print(f"   {method}: {count} ({(count/len(df)*100):.1f}%)")

if 'source' in df.columns:
    print(f"\n📰 SOURCE DIVERSITY:")
    print(f"   Unique sources: {df['source'].nunique()}")

if 'published_dt' in locals() or 'published' in df.columns:
    if 'published_dt' in df.columns:
        date_col = 'published_dt'
    else:
        date_col = 'published'
        df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    
    valid_dates = df[date_col].notna()
    if valid_dates.any():
        print(f"\n📅 TEMPORAL COVERAGE:")
        print(f"   From: {df[date_col].min()}")
        print(f"   To: {df[date_col].max()}")
        print(f"   Total days: {(df[date_col].max() - df[date_col].min()).days + 1}")

if 'word_count' in df.columns:
    print(f"\n📝 TEXT STATISTICS:")
    print(f"   Average word count: {df['word_count'].mean():.1f}")
    print(f"   Median word count: {df['word_count'].median():.1f}")
    print(f"   Articles with < 50 words: {(df['word_count'] < 50).sum()}")
    print(f"   Articles with > 500 words: {(df['word_count'] > 500).sum()}")

if 'has_financial_content' in df.columns:
    fin_articles = df['has_financial_content'].sum()
    print(f"\n💰 FINANCIAL CONTENT:")
    print(f"   Articles with financial terms: {fin_articles} ({(fin_articles/len(df)*100):.1f}%)")

if 'data_quality_score' in df.columns:
    print(f"\n⭐ QUALITY METRICS:")
    print(f"   Average quality score: {df['data_quality_score'].mean():.3f}")
    high_quality = (df['data_quality_score'] > 0.7).sum()
    print(f"   High quality articles (>0.7): {high_quality} ({(high_quality/len(df)*100):.1f}%)")

if 'ready_for_analysis' in df.columns:
    ready = df['ready_for_analysis'].sum()
    print(f"\n✅ READY FOR SENTIMENT ANALYSIS:")
    print(f"   Articles ready: {ready} ({(ready/len(df)*100):.1f}%)")
    print(f"   Articles needing more preprocessing: {len(df) - ready}")

print("\n" + "=" * 70)
print("EDA COMPLETE - READY FOR SENTIMENT ANALYSIS")
print("=" * 70)

## 14. Next Steps

In [ ]:
print("\n🚀 RECOMMENDED NEXT STEPS:")
print("-" * 50)
print("1. Run sentiment analysis using FinBERT")
print("2. Focus on articles marked as 'ready_for_analysis'")
print("3. Compare sentiment across different sources")
print("4. Analyze sentiment trends over time")
print("5. Generate summaries for top articles")
print("6. Create interactive dashboard")

if 'ready_for_analysis' in df.columns:
    ready_count = df['ready_for_analysis'].sum()
    if ready_count < 10:
        print("\n⚠️  WARNING: Very few articles ready for analysis.")
        print("   Consider:")
        print("   - Running the collector again to get more data")
        print("   - Loosening the quality criteria")
        print("   - Using all articles regardless of quality")